# Regularization Techniques

## Objectives
- Understand Dropout and BatchNorm
- Learn L1/L2 regularization and weight decay
- Prevent overfitting with regularization
- Visualize regularization effects
- Choose appropriate regularization strategies

## Introduction
Regularization prevents overfitting by constraining model capacity. This notebook covers the most important regularization techniques.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
## 1. Create Overfit-Prone Problem

# Generate small dataset
X, y = make_moons(n_samples=100, noise=0.3, random_state=42)
X = StandardScaler().fit_transform(X)

X_train, X_test = X[:80], X[80:]
y_train, y_test = y[:80], y[80:]

X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor(y_test)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# DataLoader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 2. Model without Regularization

class SimpleNet(nn.Module):
    def __init__(self, hidden_sizes=[128, 128, 128]):
        super().__init__()
        layers = []
        prev_size = 2
        
# Iterate through batches of data
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 2))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model_no_reg = SimpleNet()
print(f"Model: {model_no_reg}")

# Count parameters
# Update model parameters based on computed gradients
# Iterate through batches of data
total_params = sum(p.numel() for p in model_no_reg.parameters())
print(f"Total parameters: {total_params:,}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 3. Model with Dropout

class NetWithDropout(nn.Module):
    def __init__(self, hidden_sizes=[128, 128, 128], dropout_rate=0.5):
        super().__init__()
        layers = []
        prev_size = 2
        
# Iterate through batches of data
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))  # Add dropout
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 2))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model_dropout = NetWithDropout(dropout_rate=0.5)
print("Model with Dropout created.")

# Demonstrate dropout behavior
x_sample = torch.randn(5, 2)

model_dropout.train()
out_train1 = model_dropout(x_sample)
out_train2 = model_dropout(x_sample)

model_dropout.eval()
out_eval1 = model_dropout(x_sample)
out_eval2 = model_dropout(x_sample)

print(f"\nTraining mode (different outputs due to dropout):")
print(f"Output 1 and 2 equal: {torch.allclose(out_train1, out_train2)}")

print(f"\nEval mode (same outputs, dropout disabled):")
print(f"Output 1 and 2 equal: {torch.allclose(out_eval1, out_eval2)}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 4. Model with BatchNorm

class NetWithBatchNorm(nn.Module):
    def __init__(self, hidden_sizes=[128, 128, 128]):
        super().__init__()
        layers = []
        prev_size = 2
        
# Iterate through batches of data
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))  # Add BatchNorm
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 2))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model_batchnorm = NetWithBatchNorm()
print("Model with BatchNorm created.")

# Demonstrate activation distribution
x_sample = torch.randn(100, 2)

# Get first hidden layer activations
with torch.no_grad():
    # Without BatchNorm
    out_no_norm = model_no_reg[0](x_sample)  # Linear layer output
    
    # With BatchNorm
    out_bn = model_batchnorm[1](model_batchnorm[0](x_sample))  # Linear + BatchNorm

print(f"\nWithout BatchNorm:")
print(f"  Mean: {out_no_norm.mean(dim=0).mean():.4f}")
print(f"  Std: {out_no_norm.std(dim=0).mean():.4f}")

print(f"\nWith BatchNorm:")
print(f"  Mean: {out_bn.mean(dim=0).mean():.4f}")
print(f"  Std: {out_bn.std(dim=0).mean():.4f}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 5. L1 and L2 Regularization

class NetWithL2(nn.Module):
    def __init__(self, hidden_sizes=[128, 128, 128]):
        super().__init__()
        layers = []
        prev_size = 2
        
# Iterate through batches of data
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 2))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)
    
# Update model parameters based on computed gradients
    def l2_penalty(self, weight_decay=0.01):
        """Compute L2 regularization penalty"""
        penalty = 0
# Iterate through batches of data
        for param in self.parameters():
# Update model parameters based on computed gradients
            penalty += weight_decay * torch.sum(param ** 2)
        return penalty
    
# Update model parameters based on computed gradients
    def l1_penalty(self, weight_decay=0.01):
        """Compute L1 regularization penalty"""
        penalty = 0
# Iterate through batches of data
        for param in self.parameters():
# Update model parameters based on computed gradients
            penalty += weight_decay * torch.sum(torch.abs(param))
        return penalty

model_l2 = NetWithL2()

# Demonstrate
output = model_l2(X_train[:4])
# Compute the loss (error) between predictions and actual values
ce_loss = nn.CrossEntropyLoss()(output, y_train[:4])
# Update model parameters based on computed gradients
l2_penalty = model_l2.l2_penalty(weight_decay=0.01)
# Update model parameters based on computed gradients
l1_penalty = model_l2.l1_penalty(weight_decay=0.01)

print(f"CrossEntropy Loss: {ce_loss.item():.4f}")
print(f"L2 Penalty: {l2_penalty.item():.4f}")
print(f"L1 Penalty: {l1_penalty.item():.4f}")
print(f"Total Loss (CE + L2): {(ce_loss + l2_penalty).item():.4f}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 6. Training Comparison

def train_model(model, train_loader, X_test, y_test, epochs=100, 
# Update model parameters based on computed gradients
                 lr=0.001, weight_decay=0):
    """Train model and track metrics"""
# Update model parameters based on computed gradients
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
# Compute the loss (error) between predictions and actual values
    criterion = nn.CrossEntropyLoss()
    
# Compute the loss (error) between predictions and actual values
    train_losses = []
    train_accs = []
    test_accs = []
    
    model = model.to(device)
    X_test = X_test.to(device)
    y_test = y_test.to(device)
    
    model.train()
# Iterate through batches of data
    for epoch in range(epochs):
# Compute the loss (error) between predictions and actual values
        epoch_loss = 0
        train_correct = 0
        train_total = 0
        
# Iterate through batches of data
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
# Compute the loss (error) between predictions and actual values
            loss = criterion(logits, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
# Compute the loss (error) between predictions and actual values
            epoch_loss += loss.item()
            preds = logits.argmax(dim=1)
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_losses.append(epoch_loss / len(train_loader))
        train_accs.append(train_correct / train_total)
        
        # Test accuracy
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            test_preds = test_logits.argmax(dim=1)
            test_acc = (test_preds == y_test).float().mean()
            test_accs.append(test_acc.item())
        model.train()
    
    return train_losses, train_accs, test_accs

print("Training comparison models...")
# Train different models
results_no_reg = train_model(SimpleNet(), train_loader, X_test, y_test, epochs=100)
results_dropout = train_model(NetWithDropout(dropout_rate=0.3), train_loader, X_test, y_test, epochs=100)
# Update model parameters based on computed gradients
results_l2 = train_model(NetWithL2(), train_loader, X_test, y_test, epochs=100, weight_decay=0.01)
print("Training completed.")


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


In [ ]:
# Configure loss function and optimization algorithm
## 7. Visualize Regularization Effects

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Training loss
axes[0].plot(results_no_reg[0], label='No Regularization', linewidth=2)
axes[0].plot(results_dropout[0], label='Dropout', linewidth=2)
axes[0].plot(results_l2[0], label='L2 Regularization', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Training accuracy
axes[1].plot(results_no_reg[1], label='No Regularization', linewidth=2)
axes[1].plot(results_dropout[1], label='Dropout', linewidth=2)
axes[1].plot(results_l2[1], label='L2 Regularization', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Training Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.4, 1.05])

# Test accuracy
axes[2].plot(results_no_reg[2], label='No Regularization', linewidth=2)
axes[2].plot(results_dropout[2], label='Dropout', linewidth=2)
axes[2].plot(results_l2[2], label='L2 Regularization', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Test Accuracy')
axes[2].set_title('Test Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0.4, 1.05])

plt.tight_layout()
plt.show()

# Print final results
print("\nFinal Results (Epoch 100):")
print("="*60)
print(f"{'Method':<25} {'Train Acc':<15} {'Test Acc':<15}")
print("="*60)
print(f"{'No Regularization':<25} {results_no_reg[1][-1]:<15.4f} {results_no_reg[2][-1]:<15.4f}")
print(f"{'Dropout':<25} {results_dropout[1][-1]:<15.4f} {results_dropout[2][-1]:<15.4f}")
print(f"{'L2 Regularization':<25} {results_l2[1][-1]:<15.4f} {results_l2[2][-1]:<15.4f}")
print("="*60)


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 8. Overfitting Gap Visualization

fig, ax = plt.subplots(figsize=(12, 6))

# Iterate through batches of data
# Calculate overfitting gap (test - train for accuracy)
methods = ['No Regularization', 'Dropout', 'L2 Regularization']
results = [results_no_reg, results_dropout, results_l2]

# Iterate through batches of data
for method, result in zip(methods, results):
    train_acc = np.array(result[1])
    test_acc = np.array(result[2])
    gap = train_acc - test_acc
    ax.plot(gap, label=method, linewidth=2.5)

ax.set_xlabel('Epoch')
ax.set_ylabel('Overfitting Gap (Train - Test Accuracy)')
ax.set_title('Overfitting Gap Over Training')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

# Print gap statistics
print("\nOverfitting Gap Analysis (Final Epoch):")
print("="*50)
# Iterate through batches of data
for method, result in zip(methods, results):
    train_acc = result[1][-1]
    test_acc = result[2][-1]
    gap = train_acc - test_acc
    print(f"{method:<25}: {gap:.4f}")


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## 9. Regularization Technique Comparison

import pandas as pd

regularization_guide = {
    'Technique': ['Dropout', 'Batch Normalization', 'L1 Regularization', 'L2 Regularization', 'Early Stopping', 'Data Augmentation'],
    'How It Works': [
        'Randomly zeros activations during training',
        'Normalizes activations to zero mean/unit variance',
        'Adds sum of absolute weights to loss',
        'Adds sum of squared weights to loss',
        'Stops training when validation loss increases',
        'Creates more training examples'
    ],
    'Computational Cost': ['Low (train)', 'Medium', 'Low', 'Low', 'None', 'High'],
    'Best For': [
        'Deep networks, reduces co-adaptation',
        'Stable training, faster convergence',
        'Feature selection, sparse solutions',
        'General regularization, smooth solutions',
        'Preventing overfitting',
        'Limited training data'
    ]
}

df = pd.DataFrame(regularization_guide)
print(df.to_string(index=False))


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
## 10. Regularization Best Practices

print("\n" + "="*60)
print("Regularization Best Practices")
print("="*60)

best_practices = """
1. START SIMPLE:
   - Begin without regularization
   - Add if you observe overfitting

2. COMBINE TECHNIQUES:
   - Dropout + L2 regularization + early stopping
   - BatchNorm + moderate dropout
   - Data augmentation + dropout

3. DROPOUT:
# Iterate through batches of data
   - Use 0.2-0.5 for hidden layers
   - Don't apply to output layer
   - Disable during evaluation (model.eval())

4. L1/L2 REGULARIZATION:
   - L2 more common than L1
# Iterate through batches of data
   - Use weight_decay in optimizer for efficiency
   - Start with small values (0.0001-0.01)

5. BATCH NORMALIZATION:
   - Helps with training stability
   - Acts as implicit regularizer
   - Different behavior in train vs eval mode

6. EARLY STOPPING:
   - Monitor validation loss
   - Save best model
# Iterate through batches of data
   - Essential for preventing overfitting
"""

print(best_practices)
print("="*60)


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


## 🎯 Key Takeaways

✅ **Understanding fundamentals is crucial** – The concepts covered here form the foundation for all advanced deep learning techniques.

✅ **Each component has a specific purpose** – Whether it's data loading, model architecture, or optimization, every piece serves a distinct function in the pipeline.

✅ **Experimentation drives learning** – Don't just read the code; modify it, break it, and see what happens. That's how intuition develops.

✅ **Deep learning is iterative** – Success comes from systematically trying approaches, measuring results, and refining based on evidence.

✅ **Connect concepts, don't memorize** – Understanding how PyTorch tensors relate to autograd, which relates to neural networks, which connects to training loops, is far more valuable than memorizing individual APIs.

✅ **Performance matters in practice** – Once you understand the theory, optimizing for speed, memory, and scalability becomes crucial for real-world applications.


In [ ]:
# Configure loss function and optimization algorithm
## Key Takeaways
- Dropout randomly disables neurons during training, reducing co-adaptation
- Batch Normalization stabilizes training and acts as regularizer
- L1/L2 regularization constrains weight magnitudes
- Weight decay in optimizer is more efficient than L2 in loss
# Iterate through batches of data
- Combine multiple regularization techniques for best results

## Interview Q&A

**Q1: Why is dropout only applied during training?**
During training, dropout prevents co-adaptation by randomly removing neurons. During inference, we want to use the full network and all learned features, so dropout is disabled.

**Q2: How does Batch Normalization reduce overfitting?**
BatchNorm reduces internal covariate shift and acts as a regularizer by adding noise through batch statistics during training. It also allows higher learning rates.

**Q3: What's the difference between L1 and L2 regularization?**
# Iterate through batches of data
L1 encourages sparse solutions (many zero weights). L2 encourages small weights. L1 is useful for feature selection, while L2 is more common for general regularization.

## References
- [Dropout Paper](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf)
- [Batch Normalization Paper](https://arxiv.org/abs/1502.03167)
